# 10 · Harbor-style parallel eval runner

How to structure an evaluation as **dataset × agent × parallel
environments → report**, the shape harnesses like
[Harbor](https://github.com/harbor-framework/harbor) scale to thousands of envs.

**Key concepts**
- **Separation of concerns** — a *dataset* of tasks, a pluggable *agent* function,
  and an *execution environment* (RaitBox sandboxes). Swap any one independently.
- **Parallel isolated envs** — every task runs in its own `AsyncSandbox`
  concurrently, so throughput scales with concurrency (capped by
  `RAITBOX_MAX_SESSIONS`).
- **Report** — aggregate per-task pass/fail into a resolve-rate, the standard
  benchmark output.

**How it works:** `agent(task)` (Claude via Bedrock) proposes a solution; each is
run against hidden tests in a fresh sandbox; results are gathered into a scoreboard.

**Prerequisites:** AWS credentials + Bedrock model access.

In [ ]:
%pip install -q raitbox

### Dataset & pluggable agent
A list of tasks with hidden tests, plus a swappable `agent(task)` that generates a solution via Bedrock.

In [ ]:
import asyncio
import os
import re

import boto3

from raitbox import AsyncSandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

# --- dataset: coding tasks with hidden tests ---
DATASET = [
    {"name": "reverse_words",
     "prompt": "Implement `def reverse_words(s: str) -> str` reversing word order ('a b c' -> 'c b a').",
     "tests": "assert reverse_words('a b c') == 'c b a'; assert reverse_words('hi') == 'hi'"},
    {"name": "is_palindrome",
     "prompt": "Implement `def is_palindrome(s: str) -> bool` ignoring case and non-alphanumerics.",
     "tests": "assert is_palindrome('A man, a plan, a canal: Panama'); assert not is_palindrome('abc')"},
    {"name": "fib",
     "prompt": "Implement `def fib(n: int) -> int` returning the n-th Fibonacci number (fib(0)=0, fib(1)=1).",
     "tests": "assert fib(0)==0 and fib(1)==1 and fib(10)==55"},
    {"name": "flatten",
     "prompt": "Implement `def flatten(xs) -> list` fully flattening arbitrarily nested lists.",
     "tests": "assert flatten([1,[2,[3,4]],5]) == [1,2,3,4,5]"},
]


# --- agent: single-shot code generation via Bedrock (swap this to plug in another agent) ---
def agent(task: dict) -> str:
    resp = bedrock.converse(modelId=MODEL_ID,
                            system=[{"text": "Reply with ONLY the function definition. No prose, no fences."}],
                            messages=[{"role": "user", "content": [{"text": task["prompt"]}]}],
                            inferenceConfig={"maxTokens": 600, "temperature": 0})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (fenced[0] if fenced else text).strip()

### Parallel runner
Each task runs in its own `AsyncSandbox` concurrently; `evaluate` gathers pass/fail across the dataset.

In [ ]:
async def run_task(task: dict) -> dict:
    solution = await asyncio.to_thread(agent, task)        # generate (offload sync Bedrock call)
    program = solution + "\n" + task["tests"] + "\nprint('OK')"
    async with await AsyncSandbox.create(region_name=REGION) as sbx:   # isolated env per task
        ex = await sbx.run_code(program)
    passed = ex.error is None and "OK" in "".join(ex.logs.stdout)
    return {"name": task["name"], "passed": passed,
            "detail": "" if passed else (ex.error.value if ex.error else "assertion failed")}


async def evaluate(dataset):
    return await asyncio.gather(*(run_task(t) for t in dataset))

## Run the suite in parallel and report

In [ ]:
results = await evaluate(DATASET)

print(f"{'task':22} result")
for r in results:
    print(f"{r['name']:22} {'PASS' if r['passed'] else 'FAIL ' + r['detail'][:40]}")
resolved = sum(r["passed"] for r in results)
print(f"\nresolve-rate: {resolved}/{len(results)}")

## Recap & where to go next
You ran a small benchmark the way scalable harnesses do: dataset, pluggable agent,
parallel isolated environments, aggregated report.

**Related:** `05` (sequential scoring) · `07` (pass@k) · `15` (RL rollouts). For the
*real* Harbor, RaitBox can be implemented as a sandbox **provider** (an adapter,
not a notebook).